# ALF Core Quickstart: Active Learning on MNIST with `alf-core`

This tutorial runs a complete active learning loop on the classic **MNIST**
handwritten-digit dataset using **only `alf-core`** — no `alf-tools`, no PyTorch.

We start with a tiny labelled set (just 80 images), and at each round let the
model pick the digits it is *least sure about*, label them, and retrain. This is
the core idea of active learning: spend your labelling budget where it helps the
model most.

Every component here is a small, self-contained custom implementation, making
this a good template for bringing your own dataset, model, and acquisition
function to ALF:

- a custom `BaseDataset` that downloads MNIST and parses the raw IDX files (numpy only)
- a custom **softmax-regression** surrogate (pure numpy + scipy)
- a custom **uncertainty-sampling** acquisition function (predictive entropy)
- ALF's standard `DesignTask` to drive the loop

**Requirements:** `alf-core`, `numpy`, `scipy`, `matplotlib`

### What is active learning?

Labels are expensive (a human annotator, a wet-lab assay, a physics simulation).
Active learning reduces how many labels you need by letting the model choose
which unlabelled examples to label next, instead of labelling at random.

The loop is:
1. Train a surrogate model on the currently labelled data.
2. Score every unlabelled candidate with an **acquisition function**.
3. Label the top-scoring batch (here, via an oracle that looks up the true label).
4. Add them to the training set and repeat.

For MNIST we score candidates by **predictive entropy** — the digits the model
finds most ambiguous — and watch test accuracy climb as the labelled set grows.

### Framework components

ALF separates an active learning experiment into a few composable pieces. The
three marked *(custom)* are defined from scratch in this notebook; the rest are
standard `alf-core` classes used as-is:

1. **Dataset** (`MNISTDataset`, *custom* [`BaseDataset`](https://instadeepai.github.io/alf/api/alf_core/dataset/base_dataset/)): downloads MNIST, builds `Candidate`s, and provides ground-truth labels on query.
2. **Surrogate Model** (`SoftmaxClassifier`, *custom* [`BaseModel`](https://instadeepai.github.io/alf/api/alf_core/model/base_model/)): predicts a probability over the 10 digit classes for each image.
3. **Search Strategy** ([`DatasetSearch`](https://instadeepai.github.io/alf/api/alf_core/optimizer/search/)): the search space is the pool of still-unlabelled images.
4. **Acquisition Function** (`UncertaintySampling`, *custom* [`AcquisitionFunction`](https://instadeepai.github.io/alf/api/alf_core/optimizer/acquisition_function/)): scores each candidate by predictive entropy.
5. **Optimizer** ([`Optimizer`](https://instadeepai.github.io/alf/api/alf_core/optimizer/optimizer/)): combines search + acquisition into the ask/tell cycle.
6. **Oracle** ([`Oracle`](https://instadeepai.github.io/alf/api/alf_core/oracle/)): returns the true label for acquired images (a dataset lookup here).
7. **Task** ([`DesignTask`](https://instadeepai.github.io/alf/api/alf_core/tasks/design_task/)): orchestrates the multi-round loop.

## Installation

Ensure the `alf_core` package (plus `matplotlib` for the plots) is installed:

```
pip install alf_core matplotlib
```

## Step 1: Imports

In [ ]:
import gzip
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from alf_core import (
    AcquisitionFunction,
    BaseModel,
    Candidate,
    DatasetSearch,
    DesignTask,
    LabelledCandidates,
    Modality,
    Optimizer,
    Oracle,
    Predictions,
    State,
    Surrogate,
    TerminalStateLogger,
)
from alf_core.dataset.base_dataset import BaseDataset, BaseDatasetConfig
from alf_core.utils.enums import ProblemType
from scipy.special import softmax

## Step 2: Load MNIST as a custom dataset

MNIST is distributed as four [IDX-format](http://yann.lecun.com/exdb/mnist/)
binary files. We only need numpy and the standard library to download and parse
them — no extra dependencies. The two helpers below read the gzipped image and
label files into numpy arrays.

In [ ]:
MNIST_BASE = "https://ossci-datasets.s3.amazonaws.com/mnist/"


def _download(fname: str, cache_dir: Path) -> Path:
    """Download an MNIST file once and cache it locally."""
    path = cache_dir / fname
    if not path.exists():
        urllib.request.urlretrieve(MNIST_BASE + fname, path)
    return path


def read_images(path: Path) -> np.ndarray:
    """Parse an IDX image file into a (n_images, 784) uint8 array."""
    data = gzip.open(path, "rb").read()
    n = int.from_bytes(data[4:8], "big")
    rows = int.from_bytes(data[8:12], "big")
    cols = int.from_bytes(data[12:16], "big")
    return np.frombuffer(data[16:], dtype=np.uint8).reshape(n, rows * cols)


def read_labels(path: Path) -> np.ndarray:
    """Parse an IDX label file into a (n_labels,) uint8 array."""
    data = gzip.open(path, "rb").read()
    return np.frombuffer(data[8:], dtype=np.uint8)

A dataset in ALF subclasses [`BaseDataset`](https://instadeepai.github.io/alf/api/alf_core/dataset/base_dataset/)
and implements `load_dataset`, which returns the full labelled pool as
`LabelledCandidates`. Each image becomes a `Candidate` holding a flattened,
`[0, 1]`-scaled pixel vector with `Modality.IMAGE`.

We subsample to keep the demo fast, and add an `n_samples` field to the config.
We do **not** need to implement `query` (the oracle's label lookup): the default
`BaseDataset.query` already returns the stored label for each candidate, which is
exactly what we want for a dataset-backed oracle.

In [ ]:
class MNISTDatasetConfig(BaseDatasetConfig):
    n_samples: int = 4000  # subsample of the 60k training images, for demo speed
    cache_dir: str = "mnist_data"


class MNISTDataset(BaseDataset):
    """MNIST digits as ALF candidates, scored by their true class label."""

    config: MNISTDatasetConfig

    def load_dataset(self) -> LabelledCandidates:
        cache = Path(self.config.cache_dir)
        cache.mkdir(exist_ok=True)
        images = read_images(_download("train-images-idx3-ubyte.gz", cache))
        labels = read_labels(_download("train-labels-idx1-ubyte.gz", cache))

        rng = np.random.default_rng(self.config.seed)
        idx = rng.permutation(len(images))[: self.config.n_samples]
        x = images[idx].astype(np.float32) / 255.0  # flatten + scale to [0, 1]
        y = labels[idx].astype(int)

        candidates = [Candidate(data=xi, modality=Modality.IMAGE) for xi in x]
        return LabelledCandidates(candidates=candidates, labels=y)

Now we configure the dataset and call `setup()`, which loads the data and splits
it into train / validation / test / candidate-pool. We start with just **2%**
labelled (the initial training set) and hold out **25%** as a fixed test set;
everything else becomes the unlabelled candidate pool the loop will draw from.

Because the labels are integers `0–9`, we use `ProblemType.MULTICLASS` — ALF then
infers `num_classes = 10` automatically.

In [ ]:
config = MNISTDatasetConfig(
    name="mnist",
    modality=Modality.IMAGE,
    seed=42,
    train_ratio=0.02,  # start with 2% (~80 images) labelled
    validation_frac=0.0,
    test_ratio=0.25,  # fixed held-out test set
    problem_type=ProblemType.MULTICLASS,
    n_samples=4000,
)

dataset = MNISTDataset(config=config)
dataset.setup()

print(f"Classes:          {dataset.num_classes}")
print(f"Initial train:    {len(dataset.train_dataset)}")
print(f"Test:             {len(dataset.test_dataset)}")
print(f"Candidate pool:   {len(dataset.candidate_pool)}  (unlabelled)")

Let's look at a few digits from the initial labelled set:

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for ax, cand, label in zip(axes, dataset.train_dataset.candidates, dataset.train_dataset.labels):
    ax.imshow(cand.data.reshape(28, 28), cmap="gray")
    ax.set_title(int(label))
    ax.axis("off")
plt.tight_layout()
plt.show()

## Step 3: Define a softmax-regression model

Our surrogate is multinomial logistic regression (softmax regression) trained by
full-batch gradient descent — a few lines of numpy. For each image it outputs a
probability over the 10 classes, so `predict` returns `Predictions.means` of
shape `(n_candidates, 10)`, the format ALF expects for classification.

`BaseModel` requires `featurise`, `train`, `predict`, and `sample`. We don't use
`sample` here (it's for generative search), so it just raises. ALF calls
`setup()` for us before training, which sets `self.output_dim` to the number of
classes (10).

In [ ]:
class SoftmaxClassifier(BaseModel):
    """Multinomial logistic regression in pure numpy."""

    def __init__(self, n_epochs: int = 150, lr: float = 0.5, l2: float = 1e-4) -> None:
        self.n_epochs = n_epochs
        self.lr = lr  # gradient-descent step size
        self.l2 = l2  # weight-decay strength
        self.weights: np.ndarray | None = None
        self.bias: np.ndarray | None = None

    def featurise(self, inputs: list[Candidate]) -> np.ndarray:
        return np.stack([c.data.ravel() for c in inputs]).astype(np.float64)

    def train(self, train_data: LabelledCandidates, val_data=None) -> None:
        x = self.featurise(train_data.candidates)
        y = train_data.labels.astype(int)
        n, d = x.shape
        self.weights = np.zeros((d, self.output_dim))
        self.bias = np.zeros(self.output_dim)
        onehot = np.eye(self.output_dim)[y]
        for _ in range(self.n_epochs):
            probs = softmax(x @ self.weights + self.bias, axis=1)
            grad = (probs - onehot) / n  # cross-entropy gradient
            self.weights -= self.lr * (x.T @ grad + self.l2 * self.weights)
            self.bias -= self.lr * grad.sum(axis=0)

    def predict(self, inputs: list[Candidate]) -> Predictions:
        x = self.featurise(inputs)
        probs = softmax(x @ self.weights + self.bias, axis=1)
        return Predictions(means=probs)  # shape (n, 10) class probabilities

    def sample(self, condition=None):
        raise NotImplementedError("SoftmaxClassifier does not support sampling.")

## Step 4: Define an uncertainty-sampling acquisition function

The acquisition function scores each unlabelled candidate; the optimizer then
labels the highest-scoring batch. We score by **predictive entropy** — the model
is most uncertain when its class probabilities are spread out:

$$H(x) = -\sum_{c=1}^{K} p_c(x)\,\log p_c(x)$$

High entropy → the model is torn between several digits → labelling it is
informative.

In [ ]:
class UncertaintySampling(AcquisitionFunction):
    def __call__(self, search_candidates: list[Candidate], state: State) -> LabelledCandidates:
        probs = state.surrogate.predict(search_candidates).means
        entropy = -np.sum(probs * np.log(probs + 1e-12), axis=1)
        return LabelledCandidates(candidates=search_candidates, labels=entropy)

## Step 5: Assemble and run the active learning loop

With all components defined, we wire them together. `DesignTask` runs the loop:
train surrogate → score the candidate pool → acquire the most-uncertain batch →
query the oracle for true labels → retrain → repeat. We run 8 rounds of 50
images each (400 labels total), on top of the initial 80.

In [ ]:
surrogate = Surrogate(model=SoftmaxClassifier())
optimizer = Optimizer(acquisition_fn=UncertaintySampling(), search_fn=DatasetSearch())
oracle = Oracle(scorer=dataset)

task = DesignTask(num_acq_rounds=8, acq_batch_size=50)
state = task.setup(dataset=dataset, surrogate=surrogate)
task.run(
    state=state,
    state_loggers=[TerminalStateLogger()],
    optimizer=optimizer,
    oracle=oracle,
)

## Step 6: Inspect results

`state.metrics_history` holds one `RoundMetrics` per round. We plot test accuracy
against the number of labelled images to see active learning at work: accuracy
rises as the loop spends its labelling budget on the digits the model found most
confusing.

In [ ]:
rounds = [m.metrics for m in state.metrics_history]
n_train = [m["dataset/num_train"] for m in rounds]
accuracy = [m["surrogate/test_accuracy"] for m in rounds]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(n_train, accuracy, marker="o", linewidth=2, color="#e74c3c")
ax.set_xlabel("Number of labelled images")
ax.set_ylabel("Test accuracy")
ax.set_title("Active learning on MNIST: accuracy vs. labelling budget")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Start: {accuracy[0]:.3f} accuracy with {n_train[0]} labels")
print(f"End:   {accuracy[-1]:.3f} accuracy with {n_train[-1]} labels")

We can also look at *what* the loop chose to label first. `state.history[0]` holds
the batch acquired in round 1 — the digits the initial model was least sure
about. These tend to be the messy, ambiguous, or atypically-written ones.

In [ ]:
first_batch = state.history[0]
fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for ax, cand, label in zip(axes, first_batch.candidates, first_batch.labels):
    ax.imshow(cand.data.reshape(28, 28), cmap="gray")
    ax.set_title(int(label))
    ax.axis("off")
fig.suptitle("Most-uncertain digits acquired in round 1", y=1.3)
plt.tight_layout()
plt.show()

## Next steps

- Swap in a different acquisition rule (e.g. **margin sampling** — label where the
  top two class probabilities are closest) by editing `UncertaintySampling`.
- Compare acquisition strategies head-to-head with the
  [benchmarking examples](https://github.com/instadeepai/alf/tree/main/benchmark_examples).
- Replace `SoftmaxClassifier` with a neural network from `alf-tools` (e.g. `CNNModel`).
- Try a real design problem: see the
  [Offline Design Tutorial](https://github.com/instadeepai/alf/blob/main/tutorials/experiments/offline_design_tutorial.ipynb).
- Bring your own model or dataset: see the
  [How-to / Recipes](https://instadeepai.github.io/alf/how-to/index.html).